# Credit Risk Scorecard — Bước 2: WOE / IV Encoding
**Input:** `cs_training_cleaned.csv` (output từ Bước 1)  
**Output:** `cs_woe_transformed.csv` + `woe_bins.pkl` (dùng cho Bước 3)

---
### Cấu trúc notebook
1. Lý thuyết WOE & IV (công thức + ví dụ tay)
2. Load dữ liệu & setup
3. Binning tự động với `scorecardpy`
4. Điều chỉnh binning thủ công (fine-tuning)
5. Phân tích IV — chọn & loại biến
6. Visualize WOE bins từng feature
7. Transform dữ liệu sang WOE values
8. Kiểm tra monotonicity & sanity check
9. Lưu bins + dữ liệu đã transform


## 1. Lý thuyết WOE & IV

### Weight of Evidence (WOE)
WOE đo lường "mức độ bằng chứng" của từng nhóm (bin) trong việc phân biệt Good và Bad.

$$WOE_i = \ln\left(\frac{\text{Distribution of Goods}_i}{\text{Distribution of Bads}_i}\right) = \ln\left(\frac{n_{good,i} / N_{good}}{n_{bad,i} / N_{bad}}\right)$$

| Giá trị WOE | Ý nghĩa |
|---|---|
| **WOE > 0** | Nhóm này có tỷ lệ Good **cao hơn** trung bình → ít rủi ro |
| **WOE = 0** | Nhóm này bằng trung bình tổng thể |
| **WOE < 0** | Nhóm này có tỷ lệ Bad **cao hơn** trung bình → rủi ro cao |

---

### Information Value (IV)
IV tổng hợp sức mạnh dự báo của **cả một biến** dựa trên tất cả các bin:

$$IV = \sum_{i} \left(\frac{n_{good,i}}{N_{good}} - \frac{n_{bad,i}}{N_{bad}}\right) \times WOE_i$$

| Khoảng IV | Sức mạnh dự báo | Quyết định |
|---|---|---|
| < 0.02 | Không đáng kể | Loại bỏ |
| 0.02 – 0.1 | Yếu | Cân nhắc |
| 0.1 – 0.3 | Trung bình | Giữ lại |
| 0.3 – 0.5 | Mạnh | Ưu tiên |
| > 0.5 | Quá mạnh | Nghi ngờ data leakage — kiểm tra kỹ |


### Ví dụ tính tay WOE & IV

In [ ]:
import pandas as pd
import numpy as np

# ── Ví dụ minh họa: biến "past_due_30_59" (số lần quá hạn 30-59 ngày) ───────
# Giả sử sau binning ta có bảng sau:

example_data = {
    "Bin":          ["0 lần", "1 lần", "2 lần", "3+ lần"],
    "n_total":      [120_000, 18_000,  6_000,   6_000  ],
    "n_bad":        [4_800,   2_160,   1_080,   2_160  ],
}
ex = pd.DataFrame(example_data)
ex["n_good"]       = ex["n_total"] - ex["n_bad"]
ex["bad_rate"]     = ex["n_bad"]  / ex["n_total"]
ex["dist_good"]    = ex["n_good"] / ex["n_good"].sum()
ex["dist_bad"]     = ex["n_bad"]  / ex["n_bad"].sum()

# Tránh log(0) bằng epsilon
eps = 1e-9
ex["WOE"] = np.log((ex["dist_good"] + eps) / (ex["dist_bad"] + eps))
ex["IV_component"] = (ex["dist_good"] - ex["dist_bad"]) * ex["WOE"]
total_iv = ex["IV_component"].sum()

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print(ex[["Bin","n_total","n_good","n_bad","bad_rate","dist_good","dist_bad","WOE","IV_component"]].to_string(index=False))
print(f"\n{'─'*60}")
print(f"  Total IV cho biến này: {total_iv:.4f}  → {'Mạnh ✓' if total_iv >= 0.3 else 'Trung bình'}")
print(f"\n  Diễn giải:")
print(f"  - Bin '0 lần': WOE = {ex.loc[0,'WOE']:.3f} → nhóm ít rủi ro nhất")
print(f"  - Bin '3+ lần': WOE = {ex.loc[3,'WOE']:.3f} → nhóm rủi ro cao nhất")
print(f"  - Khoảng cách WOE = {ex.loc[0,'WOE'] - ex.loc[3,'WOE']:.3f} → đây là sức mạnh phân tách")


## 2. Load dữ liệu & setup

In [ ]:
import warnings, pickle
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import scorecardpy as sc

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
pd.set_option("display.max_columns", 20)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "font.family": "sans-serif",
})

PALETTE = {"good": "#2196F3", "bad": "#E53935", "neutral": "#78909C",
           "woe_pos": "#1976D2", "woe_neg": "#C62828", "iv_bar": "#5C6BC0"}

# ── Load clean data từ Bước 1 ───────────────────────────────────────────────
df = pd.read_csv("cs_training_cleaned.csv")

# scorecardpy yêu cầu tên cột target là bất kỳ nhưng phải int 0/1
assert df["target"].isin([0, 1]).all(), "target phải là 0/1"

print(f"✓ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Bad rate: {df['target'].mean()*100:.2f}%")
print(f"  Columns : {list(df.columns)}")


## 3. Binning tự động với `scorecardpy`

`sc.woebin()` dùng thuật toán **Monotone Optimal Binning** — tự động tìm số bins tối ưu  
sao cho WOE có xu hướng đơn điệu (monotone) theo thứ tự bins.


In [ ]:
# ── Danh sách feature columns (loại trừ target và flag) ─────────────────────
FEATURE_COLS = [
    "revolving_util", "age", "past_due_30_59", "debt_ratio",
    "monthly_income", "open_credit_lines", "late_90_days",
    "real_estate_loans", "past_due_60_89", "dependents",
]

# ── Auto binning ─────────────────────────────────────────────────────────────
# breaks_adj: cho phép điều chỉnh thủ công nếu cần (để {} để tự động hoàn toàn)
print("Đang chạy auto binning... (mất khoảng 10-30 giây)")

bins_auto = sc.woebin(
    df[FEATURE_COLS + ["target"]],
    y="target",
    bin_num_limit=10,        # tối đa 10 bins mỗi biến
    positive="bad|1",        # định nghĩa nhãn "bad"
    print_step=0,            # tắt verbose output
)

print("✓ Auto binning hoàn tất")
print(f"  Số biến được bin: {len(bins_auto)}")


In [ ]:
# ── Xem kết quả binning thô cho 1 biến mẫu ──────────────────────────────────
sample_var = "past_due_30_59"
print(f"Binning kết quả cho: {sample_var}")
print(bins_auto[sample_var][
    ["bin", "count", "count_distr", "good", "bad", "badprob", "woe", "bin_iv"]
].to_string(index=False))


## 4. Điều chỉnh Binning thủ công (Fine-tuning)

Auto binning đôi khi tạo ra bins quá nhiều hoặc phá vỡ monotonicity.  
Ta điều chỉnh bằng tham số `breaks_adj` — chỉ định điểm cắt thủ công cho những biến cần chỉnh.

**Nguyên tắc điều chỉnh:**
- Mỗi bin nên có ít nhất **5% tổng số records**
- WOE nên **đơn điệu** (tăng dần hoặc giảm dần) theo thứ tự bins
- Số bins lý tưởng: **4–6 bins** cho biến liên tục


In [ ]:
# ── Định nghĩa breaks điều chỉnh thủ công ────────────────────────────────────
# Chỉ override những biến cần chỉnh — các biến còn lại giữ nguyên auto

BREAKS_ADJ = {
    # past_due_*: gom nhóm "0 lần" vs "1-2 lần" vs "3+ lần" vì ý nghĩa business rõ ràng
    "past_due_30_59": [1, 3],
    "past_due_60_89": [1, 2],
    "late_90_days":   [1, 2],

    # revolving_util: ngưỡng 30%, 60%, 90% là chuẩn trong credit scoring thực tế
    "revolving_util": [0.30, 0.60, 0.90],

    # age: nhóm theo giai đoạn cuộc sống
    "age": [25, 35, 50, 65],

    # debt_ratio: ngưỡng 0.35 và 0.50 là chuẩn DTI thông dụng
    "debt_ratio": [0.20, 0.35, 0.50],
}

print("Đang chạy fine-tuned binning...")
bins_final = sc.woebin(
    df[FEATURE_COLS + ["target"]],
    y="target",
    breaks_list=BREAKS_ADJ,
    bin_num_limit=10,
    positive="bad|1",
    print_step=0,
)
print("✓ Fine-tuned binning hoàn tất")


In [ ]:
# ── So sánh auto vs fine-tuned cho past_due_30_59 ────────────────────────────
print("=== AUTO BINNING ===")
print(bins_auto["past_due_30_59"][["bin","count","badprob","woe","bin_iv"]].to_string(index=False))

print("\n=== FINE-TUNED BINNING ===")
print(bins_final["past_due_30_59"][["bin","count","badprob","woe","bin_iv"]].to_string(index=False))


## 5. Phân tích IV — Chọn & Loại biến

In [ ]:
# ── Tổng hợp IV toàn bộ biến ────────────────────────────────────────────────
def compute_iv_summary(bins: dict) -> pd.DataFrame:
    """
    Tổng hợp IV của tất cả biến từ bins dict.
    Thêm cột: số bins, IV classification, và quyết định keep/drop.
    """
    rows = []
    for var, tbl in bins.items():
        iv_total  = tbl["bin_iv"].sum()
        n_bins    = len(tbl)
        max_woe   = tbl["woe"].max()
        min_woe   = tbl["woe"].min()
        woe_range = max_woe - min_woe

        if iv_total < 0.02:
            classification = "Negligible"
            decision = "DROP"
        elif iv_total < 0.1:
            classification = "Weak"
            decision = "CONSIDER"
        elif iv_total < 0.3:
            classification = "Medium"
            decision = "KEEP"
        elif iv_total <= 0.5:
            classification = "Strong"
            decision = "KEEP ★"
        else:
            classification = "Suspicious"
            decision = "INVESTIGATE"

        rows.append({
            "Variable":       var,
            "IV":             round(iv_total, 4),
            "# Bins":         n_bins,
            "WOE Range":      round(woe_range, 3),
            "Classification": classification,
            "Decision":       decision,
        })

    return (pd.DataFrame(rows)
              .sort_values("IV", ascending=False)
              .reset_index(drop=True))

iv_summary = compute_iv_summary(bins_final)
print(iv_summary.to_string(index=False))


In [ ]:
# ── Visualize IV bar chart ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))

colors = []
for _, row in iv_summary.iterrows():
    if row["IV"] >= 0.3:
        colors.append("#1565C0")    # mạnh — xanh đậm
    elif row["IV"] >= 0.1:
        colors.append("#42A5F5")    # trung bình — xanh nhạt
    elif row["IV"] >= 0.02:
        colors.append("#B0BEC5")    # yếu — xám
    else:
        colors.append("#EF9A9A")    # loại bỏ — đỏ nhạt

bars = ax.barh(
    iv_summary["Variable"][::-1],
    iv_summary["IV"][::-1],
    color=colors[::-1],
    height=0.6,
    edgecolor="white",
)

# Thêm giá trị IV và nhãn classification
for bar, (_, row) in zip(bars, iv_summary[::-1].iterrows()):
    ax.text(bar.get_width() + 0.003,
            bar.get_y() + bar.get_height() / 2,
            f"{row['IV']:.4f}  {row['Classification']}",
            va="center", fontsize=10)

# Vạch ngưỡng
for thresh, label, color in [
    (0.02, "Min (0.02)", "#FF8F00"),
    (0.1,  "Weak/Med",  "#F57F17"),
    (0.3,  "Med/Strong","#1B5E20"),
]:
    ax.axvline(thresh, color=color, linestyle="--", linewidth=1.2,
               alpha=0.7, label=label)

ax.set_xlabel("Information Value (IV)", fontsize=11)
ax.set_title("Information Value — Sức mạnh dự báo của từng biến", fontsize=13)
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(0, iv_summary["IV"].max() * 1.35)
plt.tight_layout()
plt.savefig("07_iv_summary.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Xác định biến sẽ giữ lại ────────────────────────────────────────────────
SELECTED_VARS = iv_summary.loc[
    iv_summary["IV"] >= 0.02, "Variable"
].tolist()

DROPPED_VARS = iv_summary.loc[
    iv_summary["IV"] < 0.02, "Variable"
].tolist()

print(f"✓ Giữ lại : {len(SELECTED_VARS)} biến → {SELECTED_VARS}")
print(f"✗ Loại bỏ : {len(DROPPED_VARS)} biến → {DROPPED_VARS}")


## 6. Visualize WOE Bins từng Feature

Mỗi chart gồm 2 phần:
- **Bar trái (xanh/đỏ):** WOE value — dương = tốt, âm = rủi ro
- **Line (cam):** Bad rate % — xu hướng monotone là lý tưởng


In [ ]:
def plot_woe_bins(bins: dict, variables: list, ncols: int = 3):
    """
    Vẽ WOE chart cho danh sách biến.
    Mỗi subplot: cột WOE (màu theo dương/âm) + đường bad rate.
    """
    nrows = int(np.ceil(len(variables) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 5.5, nrows * 4.2))
    axes = np.array(axes).flatten()

    for i, var in enumerate(variables):
        ax1 = axes[i]
        tbl = bins[var].copy()

        # Labels ngắn gọn cho trục x
        labels = tbl["bin"].astype(str).str.replace("missing", "N/A")

        # ── Bar: WOE ────────────────────────────────────────────────────────
        bar_colors = [
            PALETTE["woe_pos"] if w >= 0 else PALETTE["woe_neg"]
            for w in tbl["woe"]
        ]
        bars = ax1.bar(
            range(len(tbl)), tbl["woe"],
            color=bar_colors, alpha=0.85, width=0.55, edgecolor="white",
        )

        # Thêm giá trị WOE lên bar
        for bar, woe_val in zip(bars, tbl["woe"]):
            ypos = bar.get_height() + 0.02 if woe_val >= 0 else bar.get_height() - 0.06
            ax1.text(bar.get_x() + bar.get_width() / 2, ypos,
                     f"{woe_val:.2f}", ha="center", va="bottom", fontsize=8.5,
                     fontweight="bold")

        ax1.axhline(0, color="black", linewidth=0.8, linestyle="-")
        ax1.set_xticks(range(len(tbl)))
        ax1.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
        ax1.set_ylabel("WOE", fontsize=9, color="#1A237E")
        ax1.tick_params(axis="y", labelcolor="#1A237E", labelsize=8)

        # ── Line: Bad rate (trục phụ) ────────────────────────────────────────
        ax2 = ax1.twinx()
        ax2.plot(range(len(tbl)), tbl["badprob"] * 100,
                 color="#FF6F00", marker="o", markersize=5,
                 linewidth=2, linestyle="--", label="Bad rate %")
        ax2.set_ylabel("Bad rate %", fontsize=9, color="#E65100")
        ax2.tick_params(axis="y", labelcolor="#E65100", labelsize=8)

        # ── IV annotation ────────────────────────────────────────────────────
        iv_val = tbl["bin_iv"].sum()
        ax1.set_title(
            f"{var}  |  IV = {iv_val:.4f}",
            fontsize=10, fontweight="bold", pad=8,
        )

        # Thêm count nhỏ phía dưới mỗi bar
        for j, (cnt, distr) in enumerate(zip(tbl["count"], tbl["count_distr"])):
            ax1.text(j, ax1.get_ylim()[0],
                     f"n={cnt:,}
({distr*100:.1f}%)",
                     ha="center", va="bottom", fontsize=6.5,
                     color="gray", style="italic")

    # Tắt axes thừa
    for j in range(len(variables), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("WOE Analysis — Tất cả biến được chọn",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("08_woe_bins_all.png", bbox_inches="tight")
    plt.show()


plot_woe_bins(bins_final, SELECTED_VARS, ncols=3)


## 7. Kiểm tra Monotonicity

Một scorecard tốt cần WOE **đơn điệu** (monotone) — nghĩa là khi feature tăng thì WOE  
phải liên tục tăng hoặc liên tục giảm, không được zigzag.  
Nếu vi phạm monotonicity → cần điều chỉnh lại binning.


In [ ]:
def check_monotonicity(bins: dict, variables: list) -> pd.DataFrame:
    """
    Kiểm tra WOE có đơn điệu không cho từng biến.
    Trả về DataFrame với cột: is_monotone, direction, n_violations.
    """
    results = []
    for var in variables:
        woe_vals = bins[var]["woe"].values
        if len(woe_vals) <= 2:
            results.append({
                "Variable": var, "is_monotone": True,
                "Direction": "N/A (<=2 bins)", "Violations": 0,
            })
            continue

        diffs = np.diff(woe_vals)
        n_pos = (diffs > 0).sum()
        n_neg = (diffs < 0).sum()

        if n_neg == 0:
            direction, is_mono = "Tăng dần ↑", True
        elif n_pos == 0:
            direction, is_mono = "Giảm dần ↓", True
        else:
            violations = min(n_pos, n_neg)
            direction  = f"Không đơn điệu ({'tăng chủ đạo' if n_pos > n_neg else 'giảm chủ đạo'})"
            is_mono    = False

        results.append({
            "Variable":    var,
            "is_monotone": is_mono,
            "Direction":   direction,
            "Violations":  0 if is_mono else min(n_pos, n_neg),
        })

    df_mono = pd.DataFrame(results)
    n_pass = df_mono["is_monotone"].sum()
    print(f"Monotonicity check: {n_pass}/{len(df_mono)} biến đạt")
    print()
    print(df_mono.to_string(index=False))
    return df_mono

mono_check = check_monotonicity(bins_final, SELECTED_VARS)

# Cảnh báo nếu có biến vi phạm
violations = mono_check[~mono_check["is_monotone"]]
if len(violations) > 0:
    print(f"\n⚠️  {len(violations)} biến vi phạm monotonicity:")
    for _, row in violations.iterrows():
        print(f"   → {row['Variable']}: {row['Violations']} điểm zigzag — cân nhắc điều chỉnh breaks_adj")
else:
    print("\n✓ Tất cả biến đều đơn điệu — binning tốt!")


## 8. Transform dữ liệu sang WOE values

Sau khi đã hài lòng với bins, ta chuyển toàn bộ dataset:  
mỗi giá trị gốc được thay thế bằng WOE của bin tương ứng.


In [ ]:
# ── Transform toàn bộ dataset ────────────────────────────────────────────────
df_woe = sc.woebin_ply(df, bins_final)

# Kiểm tra kết quả
print("Shape sau transform:", df_woe.shape)
print()
print("5 dòng đầu (cột WOE):")
woe_cols = [c for c in df_woe.columns if c.endswith("_woe")]
print(df_woe[woe_cols + ["target"]].head().to_string(index=False))


In [ ]:
# ── Kiểm tra distribution của WOE values ─────────────────────────────────────
print("Thống kê mô tả WOE columns:")
woe_selected = [f"{v}_woe" for v in SELECTED_VARS if f"{v}_woe" in df_woe.columns]
print(df_woe[woe_selected].describe().round(4).to_string())
print()

# Kiểm tra không có NaN
nan_counts = df_woe[woe_selected].isnull().sum()
if nan_counts.sum() == 0:
    print("✓ Không có NaN trong WOE columns")
else:
    print("⚠️  Có NaN trong WOE columns:")
    print(nan_counts[nan_counts > 0])


## 9. Correlation giữa các WOE variables

Sau khi transform sang WOE, kiểm tra lại correlation để phát hiện **multicollinearity**.  
Nếu 2 biến có |correlation| > 0.75 → nên loại 1 (giữ biến có IV cao hơn).


In [ ]:
corr_woe = df_woe[woe_selected].corr()

# ── Heatmap ──────────────────────────────────────────────────────────────────
mask = np.triu(np.ones_like(corr_woe, dtype=bool))
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr_woe,
    mask=mask,
    annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor="white",
    annot_kws={"size": 9},
    ax=ax,
)
ax.set_title("Correlation Matrix — WOE Variables", fontsize=12, fontweight="bold")
# Rút gọn tên cột cho dễ đọc
short_labels = [c.replace("_woe", "") for c in woe_selected]
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)
plt.tight_layout()
plt.savefig("09_woe_correlation.png", bbox_inches="tight")
plt.show()

# ── Cảnh báo multicollinearity ───────────────────────────────────────────────
print("Cặp có |correlation| > 0.75 (nguy cơ multicollinearity):")
found = False
for i in range(len(corr_woe.columns)):
    for j in range(i + 1, len(corr_woe.columns)):
        val = corr_woe.iloc[i, j]
        if abs(val) > 0.75:
            c1 = corr_woe.columns[i].replace("_woe","")
            c2 = corr_woe.columns[j].replace("_woe","")
            # Tìm IV của từng biến để gợi ý loại biến nào
            iv1 = iv_summary.loc[iv_summary["Variable"]==c1, "IV"].values[0]
            iv2 = iv_summary.loc[iv_summary["Variable"]==c2, "IV"].values[0]
            keep = c1 if iv1 >= iv2 else c2
            drop = c2 if iv1 >= iv2 else c1
            print(f"  {c1} ↔ {c2}  r={val:.3f}  → Giữ {keep} (IV cao hơn), cân nhắc loại {drop}")
            found = True
if not found:
    print("  ✓ Không phát hiện multicollinearity nghiêm trọng")


## 10. Lưu bins & dữ liệu đã transform

In [ ]:
import pickle

# ── Chuẩn bị final dataset cho Bước 3 ────────────────────────────────────────
# Chỉ giữ WOE columns của SELECTED_VARS + target
final_woe_cols = [f"{v}_woe" for v in SELECTED_VARS if f"{v}_woe" in df_woe.columns]
df_model_ready = df_woe[final_woe_cols + ["target"]].copy()

# Thêm flag missing (từ Bước 1) nếu có
if "monthly_income_missing" in df.columns:
    df_model_ready["monthly_income_missing"] = df["monthly_income_missing"].values

print(f"Dataset sẵn sàng cho model: {df_model_ready.shape}")
print(f"Columns: {list(df_model_ready.columns)}")

# ── Lưu files ────────────────────────────────────────────────────────────────
df_model_ready.to_csv("cs_woe_transformed.csv", index=False)
print("\n✓ Saved: cs_woe_transformed.csv")

with open("woe_bins.pkl", "wb") as f:
    pickle.dump(bins_final, f)
print("✓ Saved: woe_bins.pkl")

with open("selected_vars.pkl", "wb") as f:
    pickle.dump(SELECTED_VARS, f)
print("✓ Saved: selected_vars.pkl")

# ── Tóm tắt bước 2 ───────────────────────────────────────────────────────────
print()
print("=" * 55)
print("  TÓM TẮT BƯỚC 2 — WOE / IV ENCODING")
print("=" * 55)
print(f"  Tổng biến ban đầu  : {len(FEATURE_COLS)}")
print(f"  Biến giữ lại (IV≥0.02): {len(SELECTED_VARS)}")
print(f"  Biến loại bỏ      : {len(DROPPED_VARS)}")
print()
print("  Top 5 biến mạnh nhất (theo IV):")
for _, row in iv_summary.head(5).iterrows():
    print(f"    {row['Variable']:35s}  IV = {row['IV']:.4f}  {row['Classification']}")
print()
print("─── Bước tiếp theo: Bước 3 — Logistic Regression Scorecard ───")
